# 16 · Final comparison of every model

This notebook puts **every model on every evaluation** in one place: the tables and figures for the paper's results section. It trains nothing and needs no GPU. It only reads the saved results, so it runs in about a minute on a laptop or on Databricks CPU compute. Rerun it whenever new results are pushed.

**What it shows**
1. **One table of every model:** puzzle accuracy with its 95% interval, legal rate and ACPL; the held-out game positions (Stockfish match, blunders); full games (score, invalid moves); and the training compute behind each model, summed over its stages.
2. **Which models really differ from `sft_v1`:** paired comparisons on the same 1,000 puzzles. With 15+ comparisons, some p-values fall below 0.05 by chance alone, so the p-values are also adjusted for the number of comparisons (Holm's method). A model is "significantly different" only if its adjusted p-value is below 0.05.
3. **Why accuracy moves:** accuracy = (share of legal answers) × (accuracy when the answer is legal). This separates "names fewer impossible moves" from "chooses better moves", which is the main story of the RL and legal-move results.
4. **Compute:** what each model cost to train, so methods can be compared at a similar budget, as our proposal promised.

Models that have no results yet are listed and skipped, so the notebook runs at any point. A model trained later is added with one line in `MODELS`.

**Running it:** click **Run all**. With `QUICK_CHECK = True`, the outputs go to a temporary folder instead of `results/`. On Databricks, commit and push only `results/` afterwards.

**Outputs** in `results/figures/final/`: `final_results` (table), `vs_sft_v1` (table), `compute` (table), `accuracy.png`, `accuracy_decomposition.png`, `game_positions.png`, `games_vs_stockfish.png`

In [ ]:
# On Databricks: install our packages and Stockfish for this session. Does nothing on other machines.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # so Databricks finds our code in src/
from chesslm.databricks import setup_databricks

setup_databricks()

In [ ]:
# Settings
QUICK_CHECK = False   # True = outputs go to a temporary folder instead of results/ (to check that everything works)
MODELS = {            # model -> (method, what changed). The order of rows in every table and figure.
    "base": ("base", "no training"),
    "base_legal_moves": ("base", "legal moves listed in the prompt"),
    "sft_v1": ("SFT", "20k positions, the player's move"),
    "sft_stockfish": ("SFT", "20k positions, Stockfish's move"),
    "sft_100k": ("SFT", "100k positions, the player's move"),
    "sft_100k_stockfish": ("SFT", "100k positions, Stockfish's move"),
    "sft_legal_moves": ("SFT", "20k positions, legal moves listed in the prompt"),
    "sft_100k_legal_moves": ("SFT", "100k positions, legal moves listed in the prompt"),
    "dpo_v1": ("DPO from sft_v1", "rejected = a random move losing 200+ cp"),
    "dpo_v2": ("DPO from sft_v1", "rejected = the model's own mistakes"),
    "dpo_v3": ("DPO from sft_v1", "own mistakes + SFT loss"),
    "grpo_v1": ("RL from sft_v1", "no KL penalty"),
    "grpo_v2": ("RL from sft_v1", "KL penalty 0.04"),
    "grpo_v3": ("RL from sft_v1", "KL penalty 0.04, reward without legality bonus"),
    "grpo_v4": ("RL from sft_v1", "KL penalty 0.1"),
    "grpo_v5": ("RL from sft_v1", "KL penalty 0.2"),
    "sft_legal_dpo": ("combination", "sft_legal_moves -> DPO (dpo_v3's setting)"),
    "sft_legal_rl": ("combination", "sft_legal_moves -> RL (KL penalty 0.2)"),
    "sft_legal_dpo_rl": ("combination", "sft_legal_moves -> DPO -> RL"),
}
REFERENCE = "sft_v1"  # every model is compared with this one

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 250)  # print wide tables on one line
pd.set_option("display.max_columns", 30)

from chesslm import experiments, plots
from chesslm.analysis import chance_correct, holm_adjust, wilson_interval
from chesslm.config import REPO_ROOT
from chesslm.experiments import compare_with, figures_dir, save_table, training_stages

FIGURES_NAME = "final"
if QUICK_CHECK:
    experiments.use_quick_check_folder()
    FIGURES_NAME += "_check"
FIGURES = figures_dir(FIGURES_NAME)

summary = pd.read_csv(experiments.RESULTS / "summary.csv").set_index("run")
positions = pd.read_csv(experiments.RESULTS / "positions_summary.csv").set_index("run")
games = pd.read_csv(experiments.RESULTS / "games_summary.csv")

present = [m for m in MODELS if m in summary.index]
missing = [m for m in MODELS if m not in summary.index]
print(f"{len(present)} models with puzzle results: {', '.join(present)}")
print(f"Not graded yet (skipped): {', '.join(missing) or 'none'}")

## 1. Every model on every evaluation

**How to read the table**
- **Puzzles** (1,000 test puzzles): accuracy with its 95% interval, the share of legal answers, and ACPL (centipawns lost by the legal answers; lower is better). A random legal move solves the share printed below.
- **Game positions** (1,000 held-out positions from games): how often the answer is Stockfish's move, and the share of legal answers that are blunders (lose 200+ centipawns).
- **Games** (50 per opponent): the score against a random mover and against Stockfish at depth 1, and the share of the model's moves in all games that were invalid (replaced by a random legal move).
- **Compute:** tokens seen in training, summed over every stage behind the model (e.g. SFT, then DPO, then RL).

A dash means that evaluation has not been run for that model yet.

In [ ]:
test_puzzles = pd.read_csv(REPO_ROOT / "data" / "eval" / "puzzles_test.csv")
CHANCE = test_puzzles.apply(chance_correct, axis=1).mean()
print(f"A random legal move solves {CHANCE:.1%} of the test puzzles (on average over all its possible moves).")


def game_stat(run, opponent, column):
    row = games[(games.run == run) & (games.opponent == opponent)]
    return row[column].iloc[0] if len(row) else None


def invalid_in_games(run):
    rows = games[games.run == run]
    return (rows.invalid_move_rate * rows.model_moves).sum() / rows.model_moves.sum() if len(rows) else None


def pct(value, digits=1):
    return None if value is None or pd.isna(value) else round(value * 100, digits)


rows = {}
for run in present:
    s = summary.loc[run]
    low, high = wilson_interval(round(s.puzzle_accuracy * s.n_puzzles), s.n_puzzles)
    p = positions.loc[run] if run in positions.index else None
    stages = training_stages(run)
    rows[run] = {
        "Method": MODELS[run][0], "What changed": MODELS[run][1],
        "Puzzle accuracy (%)": pct(s.puzzle_accuracy), "95% interval": f"{low * 100:.1f}-{high * 100:.1f}",
        "Legal (%)": pct(s.legal_rate), "ACPL (cp)": round(s.acpl),
        "Positions: same as Stockfish (%)": pct(p.stockfish_match_rate) if p is not None else None,
        "Positions: blunders among legal (%)": pct(p.blunder_rate_of_legal) if p is not None else None,
        "Score vs. random (%)": pct(game_stat(run, "random", "score")),
        "Score vs. Stockfish (%)": pct(game_stat(run, "stockfish", "score")),
        "Invalid moves in games (%)": pct(invalid_in_games(run)),
        "Training tokens (M)": round(sum(st["tokens_seen"] for st in stages) / 1e6, 2),
    }
final = pd.DataFrame(rows).T
final.index.name = "Model"
save_table(final, FIGURES, "final_results",
           f"Puzzles: 1,000 test puzzles (a random legal move solves {CHANCE:.1%}); ACPL over legal answers. Positions: "
           "1,000 held-out game positions. Games: 50 per opponent; score = (wins + draws/2) / games. Training tokens: "
           "summed over every training stage behind the model. A dash = not evaluated yet.")
display(final)

## 2. Which models really differ from `sft_v1`?

For each model, the number of puzzles only it solves and the number only `sft_v1` solves, with the sign-test p-value. The **adjusted p-value** (Holm's method) accounts for making all these comparisons at once. Below 0.05 means a real difference, not noise.

In [ ]:
others = [run for run in present if run != REFERENCE]
comparison = pd.DataFrame([compare_with(run, REFERENCE) for run in others]).set_index("model")
comparison["adjusted p-value (Holm)"] = [float(f"{p:.2g}") for p in holm_adjust(list(comparison["p-value (sign test)"]))]
comparison["Accuracy (%)"] = [pct(summary.loc[run, "puzzle_accuracy"]) for run in comparison.index]


def verdict(row):
    if row["adjusted p-value (Holm)"] >= 0.05:
        return "no significant difference"
    return "better" if row["solved only by this model"] > row["solved only by the other"] else "worse"


comparison[f"vs. {REFERENCE}"] = comparison.apply(verdict, axis=1)
comparison.index.name = "Model"
save_table(comparison, FIGURES, "vs_sft_v1",
           f"Each model vs. {REFERENCE} on the same 1,000 test puzzles. Adjusted p-value: Holm's correction for "
           f"{len(comparison)} comparisons; below 0.05 = a real difference.")
display(comparison)
for label, text in [("better", "Better than"), ("worse", "Worse than"),
                    ("no significant difference", "Not significantly different from")]:
    names = list(comparison.index[comparison[f"vs. {REFERENCE}"] == label])
    print(f"{text} {REFERENCE}: {', '.join(names) or 'none'}")

## 3. Figures

- **Accuracy with 95% intervals**, against the random-move line.
- **Accuracy decomposition:** the share of legal answers, and the accuracy when the answer is legal. A model that only learns to name possible moves rises on the left but not on the right.
- **Held-out game positions:** legal answers, Stockfish match, and blunders.
- **Games against Stockfish:** wins, draws and losses.

In [ ]:
accuracy = pd.DataFrame({run: dict(zip(["low", "high"], wilson_interval(
    round(summary.loc[run, "puzzle_accuracy"] * summary.loc[run, "n_puzzles"]), summary.loc[run, "n_puzzles"])))
    for run in present}).T
accuracy["accuracy"] = summary.loc[present, "puzzle_accuracy"]
plots.accuracy_with_intervals(accuracy, CHANCE, FIGURES / "accuracy.png",
                              title="Puzzle accuracy of every model (95% intervals)")

parts = pd.DataFrame({"legal_rate": summary.loc[present, "legal_rate"],
                      "accuracy_when_legal": summary.loc[present, "puzzle_accuracy"] / summary.loc[present, "legal_rate"]})
plots.accuracy_decomposition(parts, CHANCE, FIGURES / "accuracy_decomposition.png",
                             title="Accuracy = legal answers x accuracy when legal")

with_positions = [run for run in present if run in positions.index] + ["random legal move"]
pos_table = (positions.loc[[r for r in with_positions if r in positions.index],
                           ["legal_rate", "stockfish_match_rate", "blunder_rate_of_legal"]] * 100).round(1)
pos_table.columns = ["Legal (%)", "Same as Stockfish (%)", "Blunders among legal (%)"]
plots.metric_panels(pos_table, [(c, c.lower(), "{:.0f}") for c in pos_table.columns], FIGURES / "game_positions.png",
                    title="Held-out game positions")

vs_stockfish = games[games.opponent == "stockfish"].set_index("run")
vs_stockfish = vs_stockfish.loc[[run for run in present if run in vs_stockfish.index]]
plots.game_results(vs_stockfish, FIGURES / "games_vs_stockfish.png", title="Games against Stockfish (depth 1)")
plt.show()

## 4. Compute

Each model's training stages and their cost. Our proposal promised a similar data and compute budget for every method, so the paper should compare methods by **tokens seen**. GPU-hours depend on the GPU: `sft_v1`, `dpo_v1` and `grpo_v1` ran on a laptop GPU, `sft_v1` before a memory fix that made later runs about 5 times faster per step, and everything after them ran on a Databricks A10G. Collecting DPO pairs and grading are inference, and are not counted.

In [ ]:
compute_rows = {}
for run in present:
    stages = training_stages(run)
    if not stages:
        continue
    compute_rows[run] = {"Training stages": " -> ".join(s["stage"] for s in stages),
                         "Tokens, last stage (M)": round(stages[-1]["tokens_seen"] / 1e6, 2),
                         "Tokens, all stages (M)": round(sum(s["tokens_seen"] for s in stages) / 1e6, 2),
                         "GPU-hours, last stage": round(stages[-1]["gpu_hours"], 2),
                         "GPU-hours, all stages": round(sum(s["gpu_hours"] for s in stages), 2),
                         "GPU, last stage": stages[-1]["device"]}
compute = pd.DataFrame(compute_rows).T
compute.index.name = "Model"
save_table(compute, FIGURES, "compute",
           "Training compute of each model's stages. Compare by tokens; GPU-hours depend on the GPU, and sft_v1's are "
           "inflated (laptop GPU, before the memory fix). DPO pair collection and grading are not counted.")
display(compute)

## Summary of outputs

`results/figures/final/`:
- `final_results` (table): every model on every evaluation, with training tokens
- `vs_sft_v1` (table): paired comparisons with `sft_v1`, with Holm-adjusted p-values and a verdict
- `compute` (table): each model's training stages, tokens and GPU-hours
- `accuracy.png`, `accuracy_decomposition.png`, `game_positions.png`, `games_vs_stockfish.png`

Each table is saved as `.md` (to read on GitHub) and `.csv` (to reuse in the paper).